# GlobalForms metric pilot

Run from top to bottom. The attached draft’s §3, Table 2, §4 and Appendix A are the reference. This notebook uses in-memory fixtures and produces five small blank PDFs with the optional PDF cell. The metric checks use only Python’s standard library.

| Stage | Added to the preceding form |
|---|---|
| 1 | One field, applicant name |
| 2 | An Applicant section and direct field membership |
| 3 | A Business Office section with a reviewer field and a second completion role |
| 4 | An ISO calendar-date constraint on an added review-date field |
| 5 | A single-choice gate and an explanation required only on Yes |

Fixtures carry physical regions, logical fields, labels, section memberships, role inheritance, applicability, validation and source anchors. The local JSON vocabulary is illustrative. Anchors here use PDF rectangles in unrotated, top-left CropBox-relative points. PDF stage files are generated locally on demand. The fixture renderer derives from the reference for convenience, so synthetic success is a scorer check, not evidence of recoverability.

**Draft choices exercised here.** One-to-one matching maximizes eligible matches; ambiguous ties use sorted IDs. Logical correspondence requires all owned regions to match to one field. Labels, option values and role responsibilities use simple normalized equality; for real data, freeze accepted aliases and source-grounded equivalence. Section identity maximizes shared matched fields. Choice rules are evaluated on every reference option plus unanswered, and section rules expand to fields. Selection cardinality is scored when the source establishes it. Calendar validation is diagnostic in Track A. Split/merge, complex rule equivalence, evidence scoring, rubric-based free text, matching tie-breaks and official headline weights still need pilot decisions. Exact matching here uses a small bitmask search and is intended for the fixtures only.

In [1]:
from collections import Counter
from copy import deepcopy
from datetime import date
from functools import lru_cache
from pathlib import Path
import json
import re


def anchor(x, y, w, h):
    return dict(page=1, bbox=[x, y, w, h])


def add_field(spec, key, label, y, kind="text", section=None, options=()):
    spec["evidence"][key] = anchor(48, y, 215, 16)
    field = dict(label=label, type=kind, section=section, evidenceIds=[key],
                 regions=[], options={})
    for i, option in enumerate(options or (None,)):
        region = f"{key}_{i}"
        spec["regions"][region] = anchor(280 + i * 85, y, 14 if option else 205, 16)
        field["regions"].append(region)
        if option:
            field["options"][option] = region
    if options:
        field["cardinality"] = [1, 1]
    spec["fields"][key] = field


def fixtures():
    spec = dict(schemaVersion="pilot-0", fields={}, regions={}, evidence={},
                sections={}, rules=[], validations=[])
    add_field(spec, "name", "Applicant name", 85)
    yield deepcopy(spec)
    spec["evidence"]["applicant_heading"] = anchor(48, 57, 215, 16)
    spec["sections"]["applicant"] = dict(title="Applicant", evidenceIds=["applicant_heading"])
    spec["fields"]["name"]["section"] = "applicant"
    yield deepcopy(spec)
    spec["evidence"]["office_heading"] = anchor(48, 223, 215, 16)
    spec["sections"]["office"] = dict(title="Business Office", role="Business Office",
                                      evidenceIds=["office_heading"])
    add_field(spec, "reviewer", "Reviewer name", 250, section="office")
    yield deepcopy(spec)
    add_field(spec, "review_date", "Review date (YYYY-MM-DD)", 285, "date", "office")
    spec["validations"].append(dict(fields=["review_date"], kind="iso_calendar_date",
                                    evidenceIds=["review_date"]))
    yield deepcopy(spec)
    add_field(spec, "incident", "Incident? (select one)", 120, "choice",
              "applicant", ("No", "Yes"))
    add_field(spec, "details", "If Yes, describe the incident", 155, section="applicant")
    spec["rules"].append(dict(controller="incident", op="eq", option="Yes",
                              fields=["details"], evidenceIds=["details"]))
    yield deepcopy(spec)


stages = list(fixtures())
print(json.dumps(stages[-1]["rules"], indent=2))

[
  {
    "controller": "incident",
    "op": "eq",
    "option": "Yes",
    "fields": [
      "details"
    ],
    "evidenceIds": [
      "details"
    ]
  }
]


In [2]:
# Optional. Run with WRITE_PDFS = True to create the five fixtures.
WRITE_PDFS = False
if WRITE_PDFS:
    from reportlab.pdfgen.canvas import Canvas

    folder = Path("globalforms-fixtures")
    folder.mkdir(exist_ok=True)
    for i, spec in enumerate(stages, 1):
        path = folder / f"stage-{i}.pdf"
        pdf = Canvas(str(path), pagesize=(612, 792))
        pdf.setTitle("Synthetic review form")
        pdf.setFont("Helvetica-Bold", 15)
        pdf.drawString(48, 760, "Synthetic review form")
        pdf.setFont("Helvetica-Bold", 11)
        for section in spec["sections"].values():
            x, y, _, _ = spec["evidence"][section["evidenceIds"][0]]["bbox"]
            pdf.drawString(x, 792 - y - 12, section["title"])
        pdf.setFont("Helvetica", 10)
        for field in spec["fields"].values():
            x, y, _, _ = spec["evidence"][field["evidenceIds"][0]]["bbox"]
            pdf.drawString(x, 792 - y - 12, field["label"])
            for region in field["regions"]:
                x, y, w, h = spec["regions"][region]["bbox"]
                pdf.rect(x, 792 - y - h, w, h)
            for option, region in field["options"].items():
                x, y, w, _ = spec["regions"][region]["bbox"]
                pdf.drawString(x+w+4, 792-y-12, option)
        pdf.save()
        (folder / f"stage-{i}.json").write_text(json.dumps(spec, indent=2))
        print(path)

In [3]:
def normalize(value):
    return " ".join(str(value).casefold().split())


def matching(pred_ids, ref_ids, weight):
    refs = sorted(ref_ids)
    preds = sorted(pred_ids)
    edges = [[(j, w) for j, r in enumerate(refs) if (w := weight(p, r)) > 0]
             for p in preds]

    @lru_cache(None)
    def best(i, used):
        if i == len(preds):
            return 0, ()
        result = best(i+1, used)
        for j, w in edges[i]:
            if not used & (1 << j):
                score, pairs = best(i+1, used | (1 << j))
                result = max(result, (score+w, ((preds[i], refs[j]),) + pairs))
        return result

    return dict(best(0, 0)[1])


def hit(pred, ref):
    if pred["page"] != ref["page"]:
        return False
    x, y, w, h = pred["bbox"]
    X, Y, W, H = ref["bbox"]
    if min(w, h, W, H) <= 0:
        return False
    intersection = max(0, min(x+w, X+W)-max(x, X)) * max(0, min(y+h, Y+H)-max(y, Y))
    return intersection >= .50 * W * H and intersection >= .20 * w * h


def align(pred, ref):
    pr, rr = pred["regions"], ref["regions"]
    regions = matching(pr, rr, lambda p, r: hit(pr[p], rr[r]))
    pf, rf = pred["fields"], ref["fields"]

    def same_grounding(p, r):
        owned = pf[p]["regions"]
        return bool(owned) and len(owned) == len(rf[r]["regions"]) and (
            {regions.get(k) for k in owned} == set(rf[r]["regions"])
        )

    return regions, matching(pf, rf, same_grounding)


def counts(tp, predicted, reference):
    return tp, predicted - tp, reference - tp


def f1(triple):
    tp, fp, fn = triple
    return 2*tp / (2*tp+fp+fn) if tp or fp or fn else None


def inherited_role(spec, field):
    role = field.get("role")
    section = field.get("section")
    while not role and section is not None:
        node = spec["sections"][section]
        role = node.get("role")
        section = node.get("parent")
    return normalize(role or "applicant")


def memberships(pred, ref, fields, identity=lambda p, r: True):
    shared = Counter((pred[p], ref[r]) for p, r in fields.items() if p in pred and r in ref)
    aligned = matching(set(pred.values()), set(ref.values()),
                       lambda p, r: shared[p, r] if identity(p, r) else 0)
    return counts(sum(shared[p, r] for p, r in aligned.items()), len(pred), len(ref))


def direct_sections(spec):
    return {k: f["section"] for k, f in spec["fields"].items() if f.get("section")}


def in_section(spec, field, targets):
    section = field.get("section")
    while section is not None:
        if section in targets:
            return True
        section = spec["sections"][section].get("parent")
    return False


def rule_edges(spec, ref, alignment):
    # One edge per target, with the full truth table. Unmatched edges remain errors.
    result = Counter()
    for rule in spec["rules"]:
        source = rule["controller"]
        controller = alignment.get(source, ("unmatched", source))
        options = ref["fields"].get(controller, spec["fields"][source])["options"]
        state = (None,) + tuple(sorted(options))
        if rule["op"] not in {"eq", "ne"}:
            raise ValueError("This pilot supports equality and inequality choice rules")
        outputs = tuple("undetermined" if v is None else
                        "applicable" if (v == rule["option"]) == (rule["op"] == "eq")
                        else "inactive" for v in state)
        targets = set(rule.get("fields", ()))
        targets.update(k for k, f in spec["fields"].items()
                       if in_section(spec, f, rule.get("sections", ())))
        for target in targets:
            result[controller, alignment.get(target, ("unmatched", target)), outputs] += 1
    return result


def track_a(pred, ref):
    regions, fields = align(pred, ref)
    pf, rf = pred["fields"], ref["fields"]

    def checks(p, r):
        a, b = pf[p], rf[r]
        return {
            "type": a["type"] == b["type"],
            "label": normalize(a["label"]) == normalize(b["label"]),
            "option ownership": {k: regions.get(v) for k, v in a["options"].items()} == b["options"],
            "cardinality": "cardinality" not in b or a.get("cardinality") == b["cardinality"],
        }

    checks_by_field = [checks(p, r) for p, r in fields.items()]
    p_rules = rule_edges(pred, ref, fields)
    r_rules = rule_edges(ref, ref, {r: r for r in rf})
    components = {
        "geometry": counts(len(regions), len(pred["regions"]), len(ref["regions"])),
        "fields": counts(sum(all(c.values()) for c in checks_by_field), len(pf), len(rf)),
        "sections": memberships(direct_sections(pred), direct_sections(ref), fields),
        "roles": memberships({k: inherited_role(pred, f) for k,f in pf.items()},
                             {k: inherited_role(ref, f) for k,f in rf.items()}, fields,
                             lambda p, r: p == r),
        "rules": counts(sum((p_rules & r_rules).values()), sum(p_rules.values()), sum(r_rules.values())),
    }
    local = {key: counts(sum(c[key] for c in checks_by_field), len(pf), len(rf))
             for key in ("type", "label", "option ownership", "cardinality")}

    def constraints(spec, ids):
        return Counter((tuple(sorted(ids.get(k, f"unmatched:{k}") for k in v["fields"])), v["kind"])
                       for v in spec["validations"])

    pv = constraints(pred, fields)
    rv = constraints(ref, {k: k for k in rf})
    local["validation"] = counts(sum((pv & rv).values()), sum(pv.values()), sum(rv.values()))
    return components, local

In [4]:
def valid_entry(entry):
    if not isinstance(entry, dict) or len(entry) != 1:
        return False
    if "status" in entry:
        return entry["status"] in ("missing", "conflicting", "invalid")
    return "value" in entry and type(entry["value"]) in (str, int, float, bool) and (
        not isinstance(entry["value"], str) or bool(entry["value"].strip())
    )


def same_answer(pred, ref, field):
    if not valid_entry(pred) or pred.keys() != ref.keys():
        return False
    if "status" in ref:
        return pred["status"] == ref["status"]
    a, b = pred["value"], ref["value"]
    if field["type"] == "date":
        try:
            return date.fromisoformat(a) == date.fromisoformat(b)
        except (TypeError, ValueError):
            return False
    if field["options"]:
        return a == b
    return normalize(a) == normalize(b)


def track_b(output, expected, ref, recovered=None):
    # Oracle binds reference IDs. End-to-end binds IDs by physical grounding.
    bound = {k: k for k in ref["fields"]} if recovered is None else align(recovered, ref)[1]
    submitted = {bound[k]: v for k, v in output.items() if k in bound}
    unmatched = sum(k not in bound for k in output)
    inactive = set(ref["fields"]) - expected.keys()
    # Only an absent key can stand in for the reference's missing status.
    accepted_omissions = {k for k, answer in expected.items()
                          if k not in submitted and answer == {"status": "missing"}}
    correct = accepted_omissions | {
        k for k, answer in expected.items()
        if same_answer(submitted.get(k), answer, ref["fields"][k])
    }
    covered = sum(valid_entry(submitted.get(k)) for k in expected)
    inactive_written = len(inactive & submitted.keys())
    extra_entries = inactive_written + unmatched
    success = len(correct) == len(expected) and extra_entries == 0
    return dict(value=(len(correct), len(expected) + extra_entries),
                coverage=(covered, len(expected)),
                inactive=(inactive_written, len(inactive)),
                activity=(covered + len(accepted_omissions) + len(inactive) - inactive_written,
                          len(ref["fields"])),
                complete=(int(success), 1), unmatched=unmatched)


def value(v):
    return {"value": v}


def status(s):
    return {"status": s}


def fixture_completer(spec, facts):
    # Deterministic stand-in for a completer. Scenario facts are separate from the spec.
    result = {}
    for key, field in spec["fields"].items():
        if any(facts.get(r["controller"]) != r["option"]
               for r in spec["rules"] if key in r.get("fields", ()) and r["op"] == "eq"):
            continue
        entry = facts.get(key)
        if entry is None:
            result[key] = status("missing")
            continue
        if any(key in v["fields"] and v["kind"] == "iso_calendar_date"
               for v in spec["validations"]):
            try:
                if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", entry):
                    raise ValueError(entry)
                date.fromisoformat(entry)
            except (TypeError, ValueError):
                result[key] = status("invalid")
                continue
        result[key] = value(entry)
    return result

In [5]:
def pool_a(results):
    pooled = {}
    for component in results[0]:
        rows = [item[component] for item in results]
        pooled[component] = tuple(map(sum, zip(*rows)))
    return pooled


def fmt(score):
    return "—" if score is None else f"{score:.3f}"


# Each row compares a degraded prediction with that stage's reference.
mistakes = []
for i, ref in enumerate(stages, 1):
    pred = deepcopy(ref)
    if i == 1:
        pred["regions"]["name_0"]["bbox"][0] += 180
    if i == 2:
        pred["fields"]["name"]["section"] = None
    if i == 3:
        del pred["sections"]["office"]["role"]
    if i == 4:
        pred["validations"].clear()
    if i == 5:
        pred["rules"][0]["option"] = "No"
    perfect, _ = track_a(ref, ref)
    assert all(fp == fn == 0 for tp, fp, fn in perfect.values())
    mistakes.append(track_a(pred, ref))
    scores, diagnostic = mistakes[-1]
    print(i, " ".join(f"{k}={fmt(f1(v))}" for k, v in scores.items()),
          "validation=" + fmt(f1(diagnostic["validation"])))

assert mistakes[0][0]["geometry"] == (0, 1, 1)
assert mistakes[1][0]["sections"] == (0, 0, 1)
assert mistakes[2][0]["roles"] == (1, 1, 1)
assert all(fp == fn == 0 for tp, fp, fn in mistakes[3][0].values())
assert mistakes[3][1]["validation"] == (0, 0, 1)
assert mistakes[4][0]["rules"] == (0, 1, 1)

ref = stages[-1]
equivalent = deepcopy(ref)
equivalent["rules"][0].update(op="ne", option="No")
del equivalent["sections"]["office"]["role"]
for f in equivalent["fields"].values():
    if f["section"] == "office":
        f["role"] = "Business Office"
assert track_a(equivalent, ref) == track_a(ref, ref)

wrong = deepcopy(ref)
wrong["fields"]["incident"]["options"] = {"No": "incident_1", "Yes": "incident_0"}
a, _ = track_a(wrong, ref)
assert a["geometry"] == (6, 0, 0) and a["fields"] == (4, 1, 1)

wrong = deepcopy(ref)
wrong["fields"]["incident"]["cardinality"] = [0, 2]
assert track_a(wrong, ref)[0]["fields"] == (4, 1, 1)

# Section aggregation is intentionally blind to a wholesale swap of section titles.
wrong = deepcopy(ref)
wrong["sections"]["office"]["title"], wrong["sections"]["applicant"]["title"] = (
    wrong["sections"]["applicant"]["title"], wrong["sections"]["office"]["title"]
)
assert track_a(wrong, ref)[0] == track_a(ref, ref)[0]

# Confirmed-negative reference for rules adds no TP; a hallucination adds a pooled FP.
ref_no_rule = deepcopy(ref)
ref_no_rule["rules"].clear()
clean_negative = track_a(ref_no_rule, ref_no_rule)[0]
bad_negative = track_a(ref, ref_no_rule)[0]
assert clean_negative["rules"] == (0, 0, 0)
assert bad_negative["rules"] == (0, 1, 0)
pooled = pool_a([track_a(ref, ref)[0], bad_negative])
assert pooled["rules"] == (1, 1, 0)
print("Pooled rules with a hallucination on a confirmed negative:", fmt(f1(pooled["rules"])))

1 geometry=0.000 fields=0.000 sections=— roles=0.000 rules=— validation=—
2 geometry=1.000 fields=1.000 sections=0.000 roles=1.000 rules=— validation=—
3 geometry=1.000 fields=1.000 sections=1.000 roles=0.500 rules=— validation=—
4 geometry=1.000 fields=1.000 sections=1.000 roles=1.000 rules=— validation=0.000
5 geometry=1.000 fields=1.000 sections=1.000 roles=1.000 rules=0.000 validation=1.000
Pooled rules with a hallucination on a confirmed negative: 0.667


In [6]:
# Reproduce Appendix A's imperfect prediction from independently encoded boxes.
appendix = deepcopy(stages[1])
for key in ("fields", "regions", "rules", "evidence"):
    appendix[key].clear()
add_field(appendix, "incident", "Incident?", 85, "choice", "applicant", ("No", "Yes"))
add_field(appendix, "date", "Date", 120, "date", "applicant")
add_field(appendix, "time", "Time", 155, "time", "applicant")
appendix["regions"]["date_0"]["bbox"] = [30, 120, 100, 20]
appendix["rules"] = [dict(controller="incident", op="eq", option="Yes", fields=["date", "time"])]
imperfect = deepcopy(appendix)
imperfect["regions"]["date_0"]["bbox"][0] = 100
imperfect["fields"]["time"]["type"] = "text"
add_field(imperfect, "extra", "Extra", 191, section="applicant")
imperfect["rules"] = [dict(controller="incident", op="eq", option="No", fields=["date"])]
expected_counts = dict(geometry=(3,2,1), fields=(1,3,2), sections=(2,2,1),
                       roles=(2,2,1), rules=(0,1,2))
assert track_a(imperfect, appendix)[0] == expected_counts
print("Appendix A:", {k: fmt(f1(v)) for k, v in expected_counts.items()})

Appendix A: {'geometry': '0.667', 'fields': '0.286', 'sections': '0.571', 'roles': '0.571', 'rules': '0.000'}


## Track B scoring

After grounding-based alignment in B-end-to-end, each correct expected value or input status earns one point. Answer accuracy divides this count by expected fields plus entries in inactive or unmatched fields. When the reference marks an applicable answer as missing, either an omitted key or an explicit missing status earns credit. An omitted supplied answer or an incorrect answer earns zero points. A case passes only when every expected answer is correct and there are no extra entries. Each mode ranks by complete-case success rate, then answer accuracy. Coverage, active-field accuracy and inactive-entry rate remain diagnostics. Coverage measures explicit entries. Active-field accuracy also credits accepted omissions.

In [7]:
facts = dict(name="Alex", reviewer="Sam", review_date="2004-02-29",
             incident="Yes", details="Broken latch")
gold = {k: value(v) for k,v in facts.items()}
assert fixture_completer(ref, facts) == gold

no_facts = dict(facts, incident="No")  # deliberately retains the detail
no_gold = dict(gold, incident=value("No"))
del no_gold["details"]
invalid_facts = dict(facts, review_date="2004-02-30")
invalid_gold = dict(gold, review_date=status("invalid"))

date_outputs = (fixture_completer(ref, facts), fixture_completer(ref, invalid_facts))
branch_outputs = (fixture_completer(ref, facts), fixture_completer(ref, no_facts))


def paired_pass(outputs, references, targets):
    return all(all(output.get(k) == reference.get(k) for k in targets)
               for output, reference in zip(outputs, references, strict=True))


assert paired_pass(date_outputs, (gold, invalid_gold), ["review_date"])
assert paired_pass(branch_outputs, (gold, no_gold), ["incident", "details"])

no_constraint = deepcopy(ref)
no_constraint["validations"].clear()
unsafe = fixture_completer(no_constraint, invalid_facts)
assert track_a(no_constraint, ref)[0] == track_a(ref, ref)[0]
assert not paired_pass((date_outputs[0], unsafe), (gold, invalid_gold), ["review_date"])
assert track_b(unsafe, invalid_gold, ref, no_constraint)["value"] == (4, 5)

leak = dict(no_gold, details=value("Broken latch"))
assert track_b(leak, no_gold, ref)["inactive"] == (1, 1)
assert track_b(leak, no_gold, ref)["complete"] == (0, 1)
assert track_b(leak, no_gold, ref)["value"] == (4, 5)

swap = dict(gold, name=gold["reviewer"], reviewer=gold["name"])
assert track_b(swap, gold, ref)["value"] == (3, 5)
assert track_b(swap, gold, ref)["coverage"] == (5, 5)
assert track_b(swap, gold, ref)["activity"] == (5, 5)

for input_status in ("missing", "conflicting"):
    answer = dict(gold, reviewer=status(input_status))
    assert track_b(answer, answer, ref)["complete"] == (1, 1)
    assert track_b(gold, answer, ref)["complete"] == (0, 1)

renamed = deepcopy(ref)
renamed["fields"]["applicant_full_name"] = renamed["fields"].pop("name")
renamed_answer = {("applicant_full_name" if k == "name" else k): v for k,v in gold.items()}
assert track_b(renamed_answer, gold, ref, renamed)["complete"] == (1, 1)
assert track_b(renamed_answer, gold, ref)["unmatched"] == 1

missing = dict(gold)
del missing["reviewer"]
extra = dict(gold, extra=value("extra"))

clean_score = track_b(gold, gold, ref)
assert clean_score["value"] == (5, 5) and clean_score["complete"] == (1, 1)

# Use the same output for two scenarios with different reference answers.
missing_gold = dict(gold, reviewer=status("missing"))
valid_omission = track_b(missing, missing_gold, ref)
assert valid_omission["value"] == valid_omission["activity"] == (5, 5)
assert valid_omission["complete"] == (1, 1) and valid_omission["coverage"] == (4, 5)
assert track_b(dict(missing, reviewer=None), missing_gold, ref)["complete"] == (0, 1)
assert track_b(gold, missing_gold, ref)["value"] == (4, 5)

missing_score = track_b(missing, gold, ref)
assert missing_score["value"] == missing_score["coverage"] == (4, 5)
assert missing_score["complete"] == (0, 1) and missing_score["unmatched"] == 0

extra_score = track_b(extra, gold, ref)
assert extra_score["value"] == (5, 6) and extra_score["complete"] == (0, 1)
assert extra_score["coverage"] == (5, 5) and extra_score["unmatched"] == 1
assert track_b(dict(leak, extra=value("extra")), no_gold, ref)["value"] == (4, 6)

for label, output, expected in [
    ("perfect", gold, gold), ("wrong destination", swap, gold),
    ("unavailable answer omitted", missing, missing_gold),
    ("supplied answer omitted", missing, gold), ("inactive entry", leak, no_gold),
    ("invalid date submitted", unsafe, invalid_gold),
    ("unknown extra entry", extra, gold),
]:
    result = track_b(output, expected, ref)
    correct, total = result["value"]
    case = "pass" if result["complete"][0] else "fail"
    print(f"{label}: {case}, answer accuracy {correct}/{total} ({correct / total:.1%})"
          f" | coverage {result['coverage']}, active-field accuracy {result['activity']},"
          f" inactive-entry rate {result['inactive']}, unmatched entries {result['unmatched']}")

perfect: pass, answer accuracy 5/5 (100.0%) | coverage (5, 5), active-field accuracy (5, 5), inactive-entry rate (0, 0), unmatched entries 0
wrong destination: fail, answer accuracy 3/5 (60.0%) | coverage (5, 5), active-field accuracy (5, 5), inactive-entry rate (0, 0), unmatched entries 0
unavailable answer omitted: pass, answer accuracy 5/5 (100.0%) | coverage (4, 5), active-field accuracy (5, 5), inactive-entry rate (0, 0), unmatched entries 0
supplied answer omitted: fail, answer accuracy 4/5 (80.0%) | coverage (4, 5), active-field accuracy (4, 5), inactive-entry rate (0, 0), unmatched entries 0
inactive entry: fail, answer accuracy 4/5 (80.0%) | coverage (4, 4), active-field accuracy (4, 5), inactive-entry rate (1, 1), unmatched entries 0
invalid date submitted: fail, answer accuracy 4/5 (80.0%) | coverage (5, 5), active-field accuracy (5, 5), inactive-entry rate (0, 0), unmatched entries 0
unknown extra entry: fail, answer accuracy 5/6 (83.3%) | coverage (5, 5), active-field accu

## Where the metrics may disagree with reviewers

| Comparison | Expected metric response | Human judgment to collect |
|---|---|---|
| Exact versus displaced answer box | Geometry, joint fields and downstream correspondence can all fall | Can a person still tell where to write? Could downstream use place an answer correctly? |
| Identical boxes, swapped option ownership | Geometry ties, joint fields falls | Would either wrong mapping make the form unusable? |
| Correct memberships, swapped section titles | Current section component ties | Does the task require correct section identity as conveyed to an agent or reviewer? |
| Inherited office role versus per-field office roles | Track A ties | Are the responsibilities identical? |
| Equivalent Yes / not-No rule | Track A ties when unanswered is undetermined in both | Would the same follow-up be asked when the choice is missing? |
| Missing calendar constraint | Ranked Track A ties, constraint diagnostic drops, B can accept an invalid answer and fail the case | Is successful recovery adequate for safe completion? |
| Correct expected values plus inactive-field leakage or extra entry | B answer accuracy falls and the case fails | How severe is entering the extra information? |
| Wrong destination with full coverage | B coverage and active-field accuracy tie, answer accuracy falls and the case fails | Would this be detected in review and how costly is it? |

These tests establish behavior on planted errors. Human alignment requires independent judgments. Before showing scores, have two reviewers compare shuffled prediction pairs for recoverability (A) and readiness to submit (B). Record better / tie, critical error yes/no, implicated field or rule and rationale. Report raw agreement between reviewers, then metric agreement and tie/reversal counts against adjudicated judgments, with raw denominators and examples. Include clean, perturbed and actual model outputs. Compare component diagnostics separately before testing candidate Track A weights. For B, compare complete success first and answer accuracy as the tie-break. Review any critical error rewarded as a tie or a win; use examples to decide whether to change scoring or keep a diagnostic.

## Smallest real-file pilot

1. **Freeze inputs.** Start with roughly five user-supplied PDFs spanning one plain form, grouped fields, a second role, explicit validation and a conditional branch. One form can cover multiple features. Store PDF SHA-256, source URL if available, annotation version and page coordinates. Include one confirmed-negative rule example.
2. **Author the reference independently.** Annotate JSON plus evidence rectangles or text spans. A second reviewer checks every page and distinguishes explicit, inferred and unresolved logic. Keep unresolved items out of ranked decisions until their treatment is defined. Write paired scenario narratives for the same forms, including missing and conflicting input. Keep each pair to one changed fact.
3. **Run one predictor through a thin adapter.** Entry points are `predict_spec(pdf) -> JSON` and `complete(spec, narrative) -> JSON`. Render page images for one vision-capable model such as Qwen, ask for schema-constrained output, and convert coordinates back to PDF points. Save raw JSON, parse failures and model settings. If reusing the PDF forms compiler or forms-agent repo, first check which entry point already yields regions, field labels or widgets; adapt that output to the same contract. Measure absent properties instead of copying them from the reference. The stub in this notebook exercises the evaluation path without a model call.
4. **Validate output before scoring.** Implement small structural checks for unique IDs, finite positive rectangles, known pages, ownership and option links, acyclic section parents, valid targets and evidence pointers. Log malformed or failed submissions and keep them in metric denominators. For completion, run the same narratives once with reference specs and once with recovered specs; preserve raw submitted IDs until evaluation binds them.
5. **Review divergences.** Pool counts across documents and scenarios before computing scores. Add document-macro diagnostics on positive support and report confirmed-negative hallucinations plus support. Compare reviewer judgments with component scores, B success and paired passes. Keep the gold specs, raw predictions, score rows and reviewer decisions as separate files so metric changes can be replayed. Pin eligibility, matching and weights only after reviewing disagreement cases.

**Pilot exit check.** A model can emit a valid, reviewable spec from the supplied PDF alone; both B modes score from the same independently reviewed scenarios; every planted failure has an explained metric response; and remaining human/metric disagreements have a recorded scoring decision.

### Boundaries of this implementation

The notebook reports raw Track A counts and a pooled example. It leaves the weighted headline unset. An unsupported component shows a dash on an empty prediction and reference. Real-set macro diagnostics and failed-output handling are in the implementation plan. Track B represents resolved branch states by the complement of the expected-field map. It uses ISO dates, literal choice values and normalized text for this fixture. The proposal’s free-text rubric and broader numeric/date normalization require a reviewed policy. The paired checks here compare exact fixture JSON on targeted fields. The completion stub only exercises direct equality rules and the date constraint in these staged examples.